# 06. Long-Term Memory & Skills

## Learning Objectives
- Implement long-term memory with `CompositeBackend` + `StoreBackend`
- Understand the cross-thread memory-sharing pattern
- Inject agent context through `AGENTS.md`
- Understand the structure of skills (`SKILL.md`) and Progressive Disclosure
- Understand the difference between Skills and Memory


In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("Environment setup complete")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
# Configure the OpenAI gpt-4.1 model
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


---
## 1. Why Long-Term Memory Matters

A default `StateBackend` agent **forgets everything when the conversation thread ends**.  
But a useful assistant often needs to preserve information **across threads**, such as:

- user preferences (coding style, preferred language)
- project conventions (architecture decisions, naming rules)
- feedback learned in previous conversations
- frequently referenced information (API docs, configuration values)

### Solution: `CompositeBackend`

![CompositeBackend routing](../assets/images/composite_backend.png)

Files stored under `/memories/` can be accessed **from any conversation thread**.


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend, FilesystemBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. Create a store and a checkpointer
store = InMemoryStore()          # Development only (production: PostgresStore)
checkpointer = MemorySaver()     # Preserve agent state


# 2. Composite backend factory — persist only /memories/, keep the rest ephemeral
def memory_backend_factory(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),
        },
    )


# 3. Create the agent
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""You are a personal assistant.
Save information the user wants to remember under /memories/.
If there is previously saved memory, use it when responding.
Respond in English.""",
    backend=memory_backend_factory,
    store=store,
    checkpointer=checkpointer,
)

print("Long-term memory agent created")


---
## 2. Cross-Thread Memory Sharing

Data stored in `StoreBackend` is **shared across threads**.  
In the example below, thread 1 stores preferences and thread 2 reads them later.


In [ ]:
# Thread 1: save user preferences
config_t1 = {"configurable": {"thread_id": "memory-thread-1"}}

result1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": """
Please remember the following information:
- Preferred programming language: Python
- Preferred editor: VS Code
- Coding style: Black formatter, type hints required
Save it to /memories/preferences.txt.
"""}]},
    config={**config_t1, **lf_config},
)

print("[Thread 1] Save result:")
print(result1["messages"][-1].content)


In [ ]:
# Thread 2: use the memory from another conversation
config_t2 = {"configurable": {"thread_id": "memory-thread-2"}}

result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Read /memories/preferences.txt and tell me my preferences."}]},
    config={**config_t2, **lf_config},
)

print("[Thread 2] Memory read result:")
print(result2["messages"][-1].content)


---
## 3. Injecting Context with `AGENTS.md`

If you use the `memory` parameter, the agent automatically loads **`AGENTS.md` files at startup** and injects them into the system prompt.

### What is `AGENTS.md`?
It is a Markdown file that contains **rules, conventions, and context information** that should always apply to the agent.

### Characteristics
- Always loaded when the agent starts (not on demand)
- Injected into the system prompt through `<agent_memory>`
- You can specify multiple memory sources
- The agent can update `AGENTS.md` itself with `edit_file`


In [ ]:
import tempfile

# Create a temporary directory to use as the root_dir for FilesystemBackend
tmp_dir = tempfile.mkdtemp()
print(f"Temporary directory created: {tmp_dir}")


In [ ]:
# Load AGENTS.md through the memory parameter
memory_inject_agent = create_deep_agent(
    model=model,
    system_prompt="You are a coding assistant.",
    backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    memory=["/memory/AGENTS.md"],
)

# Check whether the agent follows the rules in AGENTS.md
result = memory_inject_agent.invoke(
    {"messages": [{"role": "user", "content": "Please create a simple HTTP client class. Follow the project conventions."}]},
    config=lf_config,
)

print(result["messages"][-1].content)


---
## 4. Skills

Skills are modular instruction bundles that give the agent **specialized domain knowledge**.

### Memory vs Skills

| Comparison | Memory (`AGENTS.md`) | Skills (`SKILL.md`) |
|------|-------------------|-------------------|
| **Loading** | Always loaded | Loaded only when needed |
| **File format** | `AGENTS.md` | `SKILL.md` (YAML frontmatter) |
| **Best fit** | Rules and conventions that always apply | Large context needed for specific tasks |
| **Token efficiency** | Always consumes tokens | Saves tokens through Progressive Disclosure |
| **Size** | Best kept concise | Can be large (up to 10 MB) |
| **Updates** | Can be edited by the agent | Usually static |

### Progressive Disclosure

Skills are not fully loaded all at once:
1. At first, only the **frontmatter** (name, description, metadata) is loaded
2. The agent decides which skills are relevant to the user's request
3. Only then is the **full content** of the necessary skills loaded

This approach saves tokens while still giving the agent access to deep task-specific knowledge.


### `SKILL.md` Structure

```yaml
---
name: web-research
description: >
  Step-by-step guide for structured web research.
  Covers information gathering, verification, and summarization.
license: MIT
compatibility: Python 3.8+
metadata:
  category: research
allowed-tools: ls read_file write_file
---

# Web Research Skill

## When to use it
- When the user asks for research on a topic
- When the request depends on current information

## Workflow
1. Design search queries
2. Gather information from multiple sources
3. Cross-check the information
4. Write a structured report
```


In [ ]:
# Create an agent that uses skills
skilled_agent = create_deep_agent(
    model=model,
    system_prompt="You are a senior developer. Use the available skills to complete the task.",
    backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    skills=["/skills/"],
)

print("Skill-enabled agent created")


In [ ]:
# Ask the agent to use its skills during a code review
result = skilled_agent.invoke(
    {"messages": [{"role": "user", "content": """
Please review the following code:

```python
import os

def process_data(data):
    result = []
    for item in data:
        if item['type'] == 'user':
            name = item['name']
            email = item['email']
            query = f"INSERT INTO users VALUES ('{name}', '{email}')"
            db.execute(query)
            result.append(item)
    return result
```
"""}]},
    config=lf_config,
)

print(result["messages"][-1].content)


---
## 5. Skill Source Priority

If you specify multiple skill sources, the **later source wins**.

```python
skills=[
    "/skills/base/",     # base skills
    "/skills/user/",     # can override base
    "/skills/project/",  # highest priority
]
```

If the same skill name exists in multiple locations, the version from the last source is used.


---
## 6. Skill Inheritance for Subagents

| Subagent Type | Skill Inheritance |
|-------------------|----------|
| Built-in general-purpose subagent | **Automatically inherits** the main agent's skills |
| Custom `SubAgent` | Requires an explicit `skills` parameter |

```python
subagent = {
    "name": "reviewer",
    "description": "Code review specialist",
    "system_prompt": "...",
    "tools": [],
    "skills": ["/skills/code-review/"],
}
```


---
## Summary

| Item | Description |
|------|------|
| Long-term memory | Persist `/memories/` with `CompositeBackend` + `StoreBackend` |
| `AGENTS.md` | `memory=["/path/AGENTS.md"]` → always injected into the system prompt |
| Skills | `skills=["/skills/"]` → `SKILL.md` with Progressive Disclosure |
| Progressive Disclosure | Load frontmatter first → load full skill only when needed |
| Skill priority | Later sources win |
| Memory vs Skills | Memory = always loaded / Skills = loaded on demand |

## Next Steps
→ **[07_advanced.ipynb](./07_advanced.ipynb)**: learn advanced features such as Human-in-the-Loop, streaming, sandboxes, and ACP.
